# System Dependencies

To get started with Unstructured.io, we need a few system-wide dependencies: 

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr) 
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [ ]:
# for mac
# !brew install poppler tesseract libmagic

In [ ]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-ollama
%pip install -Uq python_dotenv
%pip install -Uq rank_bm25 
%pip install langchain-classic


In [ ]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever  

In [ ]:
import os

def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Process all PDFs in the docs folder
docs_folder = "/Users/yogeshmeka/Desktop/MULTI-MODAL-RAG/docs"

all_elements = []

for filename in os.listdir(docs_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(docs_folder, filename)
        elements = partition_document(file_path)
        
        # Tag each element with its source filename
        for el in elements:
            el.metadata.source_file = filename
        
        all_elements.extend(elements)

print(f"\n🎉 Total elements across all documents: {len(all_elements)}")

In [ ]:
# elements
#len(all_elements)


all_elements

In [ ]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in all_elements])

In [ ]:
all_elements[36].to_dict()

In [ ]:
# Gather all images
images = [all_elements for all_elements in all_elements if all_elements.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

In [ ]:
# Gather all table
tables = [all_elements for all_elements in all_elements if all_elements.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

In [ ]:
from unstructured.documents.elements import CompositeElement, Table

def create_chunks_by_title(all_elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        all_elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    
    # Convert standalone Table chunks to CompositeElement
    converted_chunks = []
    for chunk in chunks:
        if isinstance(chunk, Table):
            # Wrap table as CompositeElement so everything is the same type
            composite = CompositeElement(text=chunk.metadata.text_as_html or chunk.text)
            composite.metadata = chunk.metadata
            converted_chunks.append(composite)
        else:
            converted_chunks.append(chunk)
    
    # Verify all chunks are now CompositeElement
    types = set([str(type(c)) for c in converted_chunks])
    print(f"✅ Created {len(converted_chunks)} chunks")
    print(f"   Types: {types}")
    return converted_chunks

chunks = create_chunks_by_title(all_elements)

In [ ]:
# View all chunks
chunks

# All unique types
#set([str(type(chunk)) for chunk in chunks])

In [ ]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

In [ ]:
# View a single chunk
chunks[2].to_dict()

# View original elements
#chunks[11].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

In [ ]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOllama(model="llama3.2", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

In [ ]:
processed_chunks

In [ ]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

In [ ]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = OllamaEmbeddings(model="nomic-embed-text")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

In [ ]:
# ✅ NEW CELL: Build Hybrid Retriever (Vector + BM25 + RRF)

def create_hybrid_retriever(vectorstore, documents, k=5, vector_weight=0.5):
    """
    Creates a hybrid retriever that combines:
    - Vector/semantic search (ChromaDB) — good at meaning and paraphrasing
    - BM25 keyword search — good at exact terms, numbers, names, acronyms
    Results are merged using Reciprocal Rank Fusion (RRF)

    Args:
        vectorstore: Your ChromaDB instance
        documents: The same processed_chunks list used to build the vector store
        k: Number of results to retrieve from each retriever
        vector_weight: Weight for vector search (0.5 = equal weight with BM25)
    """
    print("🔍 Building hybrid retriever...")

    # 1. Vector retriever — semantic similarity
    vector_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    # 2. BM25 retriever — keyword / exact match
    # Uses the same documents already embedded in ChromaDB
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_retriever.k = k

    # 3. Ensemble retriever — merges both lists using RRF
    # weights=[vector_weight, 1-vector_weight] control the balance
    # 0.5 / 0.5 = equal weight; increase vector_weight to favour semantic search
    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[vector_weight, 1 - vector_weight]
    )

    print(f"✅ Hybrid retriever ready")
    print(f"   Vector weight: {vector_weight} | BM25 weight: {1 - vector_weight}")
    print(f"   Each retriever returns top-{k} results → RRF merges them")
    return hybrid_retriever


# Build the hybrid retriever using db and processed_chunks
hybrid_retriever = create_hybrid_retriever(
    vectorstore=db,
    documents=processed_chunks,
    k=5,
    vector_weight=0.5
)


In [ ]:
# ✅ CHANGED: now uses hybrid_retriever instead of plain vector retriever
query = "Which region shows the smallest decline in life expectancy due to COVID-19 according to the comparison table?"

# OLD (vector only):
# retriever = db.as_retriever(search_kwargs={"k": 3})

# NEW (hybrid — vector + BM25 + RRF):
retrieved_chunks = hybrid_retriever.invoke(query)

print(f"Retrieved {len(retrieved_chunks)} chunks via hybrid search")
for i, c in enumerate(retrieved_chunks):
    print(f"\n--- Chunk {i+1} preview ---")
    print(c.page_content[:200])

# Export to JSON
export_chunks_to_json(retrieved_chunks, "rag_results.json")


In [ ]:
def run_complete_ingestion_pipeline(docs_folder: str):
    """Run the complete RAG ingestion pipeline for all PDFs in a folder"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    all_elements = []
    
    # Step 1: Partition every PDF in the folder
    for filename in os.listdir(docs_folder):
        if filename.endswith(".pdf"):
            file_path = os.path.join(docs_folder, filename)
            elements = partition_document(file_path)
            
            for el in elements:
                el.metadata.source_file = filename
            
            all_elements.extend(elements)
    
    print(f"\n📦 Total elements across all documents: {len(all_elements)}")
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(all_elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db
# Run the complete pipeline

In [ ]:
db = run_complete_ingestion_pipeline("/Users/yogeshmeka/Desktop/MULTI-MODAL-RAG/docs")

In [ ]:
# ✅ CHANGED: uses hybrid_retriever instead of plain vector retriever
query = "According to the data table, what percentage of global deaths in the report are attributed to non-communicable diseases (NCDs)?"

# OLD (vector only):
# retriever = db.as_retriever(search_kwargs={"k": 3})
# chunks = retriever.invoke(query)

# NEW (hybrid — vector + BM25 + RRF):
retrieved_chunks = hybrid_retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        llm = ChatOllama(model="llama3.2", temperature=0)
        
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above.
If the documents don't contain sufficient information, say "I don't have enough information
to answer that question based on the provided documents."

ANSWER:"""

        message_content = [{"type": "text", "text": prompt_text}]
        
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."


final_answer = generate_final_answer(retrieved_chunks, query)
print(final_answer)
